# Gout LLMOps — C05–C06

Mục tiêu: chạy **GOUT_MT_001 lượt 1 → history thật → lượt 2**, đồng thời ghi token budget trước generation. Notebook này **cần GPU**. Chưa chạy lượt 3 (C07) và chưa có RAG.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
print('Uploaded:', zip_name)


In [ ]:
import zipfile, os, pathlib, shutil
work = pathlib.Path('/content/gout_c05_c06')
if work.exists():
    shutil.rmtree(work)
work.mkdir()
with zipfile.ZipFile(zip_name) as z:
    z.extractall(work)
root = work / 'gout_units_v01'
assert (root / 'unit03_chat_engine.py').exists(), 'ZIP thiếu unit03_chat_engine.py'
os.chdir(root)
print('Working dir:', os.getcwd())


In [ ]:
!python -m pip install -q -r requirements-unit02.txt


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Vào Runtime > Change runtime type > GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!python -m unittest discover -s tests -p "test_*.py" -v


In [ ]:
from pathlib import Path
from unit03_chat_engine import select_multi_case_turns
turns = select_multi_case_turns(Path('examples/unit01/questions.jsonl'), 'GOUT_MT_001', count=2)
for t in turns:
    print(t['sample_id'], '=>', t['user'])
assert [t['turn_id'] for t in turns] == [1, 2]
assert all(t['case_id'] == 'GOUT_MT_001' for t in turns)
print('Đúng cùng một MT case; không mượn output ST.')


In [ ]:
from unit03_chat_engine import run_two_turns
out = Path('runs/c05_c06_real')
if out.exists():
    shutil.rmtree(out)
result = run_two_turns(
    Path('examples/unit01/questions.jsonl'),
    'GOUT_MT_001',
    Path('configs/unit03_hf_locked_from_b.json'),
    Path('prompts/system_vi_v1.txt'),
    out,
)
print('run status:', result['manifest']['status'])


In [ ]:
import json
manifest = json.loads(Path('runs/c05_c06_real/manifest.json').read_text(encoding='utf-8'))
rows = [json.loads(x) for x in Path('runs/c05_c06_real/predictions.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]
print('manifest status:', manifest['status'])
print('c05_verified:', manifest.get('c05_verified'))
print('c06_verified:', manifest.get('c06_verified'))
print('revision:', manifest.get('backend_info',{}).get('resolved_revision'))
print('GPU:', manifest.get('backend_info',{}).get('gpu_name'))
for row in rows:
    print('\n', row['sample_id'], row['status'])
    print('question:', row['question'])
    if row['status'] == 'ok':
        print('answer:', row['answer'])
        print('input_budget:', row['call']['input_budget'])
    else:
        print('error:', row.get('error'))
        print('input_budget:', row.get('input_budget'))
assert len(rows) == 2
assert all(r['status'] == 'ok' for r in rows), 'Có lượt lỗi; xem prediction và input_budget.'
assert manifest['c05_verified'] is True
assert manifest['c06_verified'] is True


In [ ]:
t1, t2 = rows
msgs = t2['call']['messages']
print(json.dumps(msgs, ensure_ascii=False, indent=2))
assert [m['role'] for m in msgs] == ['system','user','assistant','user']
assert msgs[1]['content'] == t1['question']
assert msgs[2]['content'] == t1['answer']
assert msgs[3]['content'] == t2['question']
assert t1['case_id'] == t2['case_id'] == 'GOUT_MT_001'
print('C05 REAL OK: lượt 2 nhìn thấy chính output thật của lượt 1 cùng case.')


In [ ]:
for row in rows:
    budget = row['call']['input_budget']
    assert budget['fits'] is True
    assert budget['policy'] == 'reject_no_truncation'
    assert isinstance(budget['input_tokens'], int)
print('C06 REAL OK: token budget đã ghi cho cả hai lượt, không truncate.')


In [ ]:
archive = shutil.make_archive('/content/Gout_C05_C06_real_result', 'zip', 'runs/c05_c06_real')
print(archive)
files.download(archive)


Sau khi tải `Gout_C05_C06_real_result.zip`, gửi lại file đó. Nếu cả `c05_verified` và `c06_verified` đều `true`, bước tiếp theo là **C07: chạy trọn hội thoại 3 lượt và xử lý skipped khi một lượt phụ thuộc lỗi**.
